# Performance Analysis of CPU and GPU Architectures for Matrix Transposition
## CPU vs GPU Benchmarking and Optimization

**Project:** Performance Analysis of CPU and GPU Architectures for AI-Based Matrix Computation

**Operation:** Matrix Transposition

**Platform:** Google Colab (Tesla T4 GPU runtime)

**Technologies:** Python, NumPy, Numba (CPU JIT and CUDA), Pandas, Matplotlib

| # | Section |
|---|---|
| 1 | Introduction and Objectives |
| 2 | Environment and Hardware |
| 3 | Background: CPU vs GPU Architecture and Memory Hierarchy |
| 4 | Matrix Transposition and CPU Implementation |
| 5 | Naive GPU Implementation and Correctness Verification |
| 6 | Initial Performance Benchmark |
| 7 | Optimized GPU Implementation: Shared-Memory Tiled Transpose |
| 8 | CUDA Block and Grid Configuration |
| 9 | Matrix Size Benchmark |
| 10 | Comparison with Published Results |
| 11 | Full-Capacity Test: Large Matrix, Memory Usage and Data Transfer |
| 12 | Conclusions, Future Work and CSV Export |

## 1. Introduction and Objectives

Matrix operations are fundamental to many artificial intelligence and machine learning workloads. Their performance depends strongly on the underlying computer architecture and memory system.

This project investigates the performance of matrix transposition on CPU and GPU architectures. The study compares a CPU implementation with GPU implementations using CUDA and investigates how GPU parallelism, memory access patterns, shared memory, and thread-block configuration affect execution performance.

The project is based on the CPU vs GPU benchmarking methodology provided in the reference notebook, while extending the computational workload to matrix transposition.

### Objectives

1. To understand the architectural differences between CPUs and GPUs.
2. To implement matrix transposition on the CPU and GPU.
3. To compare the execution performance of CPU and GPU implementations.
4. To investigate the effect of GPU parallelism on matrix transposition.
5. To implement a naive GPU transpose using CUDA.
6. To develop an optimized GPU transpose using tiled/shared memory.
7. To investigate the effect of CUDA block and grid configurations.
8. To measure execution time and calculate GPU speedup.
9. To analyze memory usage and CPU-to-GPU data-transfer performance.
10. To identify the factors that influence CPU and GPU performance.
11. To compare the results with published benchmark results.

## 2. Environment and Hardware

The experiments are performed in Google Colab using a GPU-enabled runtime (**Runtime → Change runtime type → T4 GPU**). The libraries below are used for numerical computation, GPU programming, data analysis, visualization, and hardware monitoring.

In [ ]:
# 2.1 ENVIRONMENT SETUP

import os
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from numba import cuda, njit, prange, float32, get_num_threads

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Numba CUDA available:", cuda.is_available())

In [ ]:
# 2.2 HARDWARE DETECTION

if not cuda.is_available():
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU")

cpu_model = next(line.split(':')[1].strip() for line in open('/proc/cpuinfo') if 'model name' in line)
gpu = cuda.get_current_device()
gpu_name = gpu.name.decode() if isinstance(gpu.name, bytes) else gpu.name
free_vram, total_vram = cuda.current_context().get_memory_info()

print("=" * 60)
print("CPU")
print("=" * 60)
print("Model:", cpu_model)
print("Logical cores:", os.cpu_count())
print(f"System RAM: {psutil.virtual_memory().total / 1024 ** 3:.2f} GiB")

print("=" * 60)
print("GPU")
print("=" * 60)
print("Model:", gpu_name)
print("Compute capability:", gpu.compute_capability)
print("Streaming multiprocessors (SMs):", gpu.MULTIPROCESSOR_COUNT)
print("Max threads per block:", gpu.MAX_THREADS_PER_BLOCK)
print(f"Shared memory per block: {gpu.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.0f} KiB")
print(f"VRAM: {total_vram / 1024 ** 3:.2f} GiB total, {free_vram / 1024 ** 3:.2f} GiB free")

## 3. Background: CPU vs GPU Architecture and Memory Hierarchy

### CPU vs GPU architecture

CPUs use a small number of powerful cores optimized for sequential control flow and low-latency tasks. GPUs use many simpler cores to execute a large number of similar operations concurrently. Matrix transposition can therefore use GPU parallelism, although its performance depends heavily on memory-access efficiency.

### CPU memory hierarchy

1. Registers – Smallest and fastest storage available to CPU cores.
2. L1 Cache – Very fast cache located close to each CPU core.
3. L2 Cache – Larger than L1 but slightly slower.
4. L3 Cache – Shared among CPU cores and larger than L1/L2.
5. Main Memory (RAM) – Much larger but significantly slower than cache.

### GPU memory hierarchy

1. Registers – Private to individual GPU threads.
2. Shared Memory – Fast on-chip memory shared by threads within a CUDA block.
3. L1/L2 Cache – Hardware-managed cache used to reduce global memory accesses.
4. Global Memory (VRAM) – Large memory accessible by all GPU threads but with higher latency.

### Why this matters for transposition

Memory is fetched in chunks, not single values: a CPU loads a 64-byte cache line, and a GPU combines the accesses of a 32-thread warp into as few memory transactions as possible (**coalescing**). Neighbouring addresses can therefore be read almost for free once the first one is loaded, while scattered addresses each cost a separate fetch.

The experiments compare a CPU implementation, a naive GPU implementation, and an optimized GPU implementation that uses shared memory to keep every global-memory access coalesced.

## 4. Matrix Transposition and CPU Implementation

Transposing a matrix swaps its rows and columns:

$$B[j][i] = A[i][j]$$

The operation performs **no arithmetic**. Each element is read once and written once, so its speed is limited by how fast memory can move data (it is **memory-bound**), not by how fast the processor can calculate.

Matrices are stored **row by row** (row-major order). When reading `A` row by row, the writes to `B` jump down a column, and vice versa. One side of a transpose is therefore always scattered across memory, which is what makes it a good test of the memory system.

All matrices in this project are square and use `float32` (4 bytes per element).

In [ ]:
# 4.1 TEST MATRIX

N = 4096

# Fixed seed: the same input on every run
np.random.seed(42)
A = np.random.rand(N, N).astype(np.float32)

print("Matrix size:", N, "x", N)
print("Data type:", A.dtype)
print("Matrix memory:", round(A.nbytes / (1024 ** 2), 2), "MB")

In [ ]:
# 4.2 CPU IMPLEMENTATION
# @njit compiles the loop to machine code. It runs on a single CPU core;
# a multi-core version is added in Section 11.

@njit
def transpose_cpu(A):
    rows, cols = A.shape
    B = np.empty((cols, rows), dtype=A.dtype)

    for i in range(rows):
        for j in range(cols):
            B[j, i] = A[i, j]

    return B


B_cpu = transpose_cpu(A)  # first call also compiles the function

print("CPU transpose completed.")
print("Output shape:", B_cpu.shape)

## 5. Naive GPU Implementation and Correctness Verification

The naive kernel launches **one GPU thread per matrix element**. Each thread copies `A[row, col]` to `B[col, row]`.

`cuda.grid(2)` returns the thread's global `(x, y)` position. Here `row` comes from `x`, which is the index that changes fastest between neighbouring threads. So neighbouring threads read **down a column** of `A` (scattered reads) and write **along a row** of `B` (neighbouring writes). Half of the memory traffic is therefore uncoalesced.

In [ ]:
# 5.1 NAIVE GPU KERNEL

@cuda.jit
def transpose_gpu_naive(A, B):
    row, col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

In [ ]:
# 5.2 GPU MEMORY ALLOCATION AND KERNEL CONFIGURATION

# Copy the input matrix from CPU RAM to GPU VRAM once.
d_A = cuda.to_device(A)
d_B = cuda.device_array((N, N), dtype=np.float32)

threads_per_block = (16, 16)


def naive_grid(n, threads):
    # Enough blocks to cover an n x n matrix
    return ((n + threads[0] - 1) // threads[0], (n + threads[1] - 1) // threads[1])


blocks_per_grid = naive_grid(N, threads_per_block)

print("Threads per block:", threads_per_block)
print("Blocks per grid:", blocks_per_grid)

In [ ]:
# 5.3 RUN NAIVE GPU TRANSPOSE AND VERIFY BOTH RESULTS

transpose_gpu_naive[blocks_per_grid, threads_per_block](d_A, d_B)
cuda.synchronize()
B_gpu = d_B.copy_to_host()

B_reference = A.T
cpu_correct = np.array_equal(B_cpu, B_reference)
gpu_correct = np.array_equal(B_gpu, B_reference)

print("CPU result correct:", cpu_correct)
print("GPU result correct:", gpu_correct)

if not (cpu_correct and gpu_correct):
    raise AssertionError('Transpose verification failed; do not continue to benchmarking.')

## 6. Initial Performance Benchmark

**Timing method**

- **CPU:** one timed run, after the function has already been compiled in Section 4.
- **GPU:** one warm-up launch, then the average of `REPS` launches. CUDA launches are asynchronous, so `cuda.synchronize()` is called before and after the timer. A single launch of a ~1 ms kernel is too noisy to report on its own.
- **Kernel-only:** the matrix is already in VRAM, so these GPU times **exclude** copying data between RAM and VRAM. Transfer times are measured separately in Section 11.

**Effective bandwidth** is reported alongside time, because transposition only moves data:

$$\text{Effective bandwidth (GB/s)} = \frac{2 \times \text{matrix size in bytes}}{\text{time}}$$

(the matrix is read once and written once).

In [ ]:
# 6.1 TIMING HELPERS

REPS = 20


def time_gpu(kernel, blocks, threads, *args, reps=REPS):
    kernel[blocks, threads](*args)  # warm-up
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        kernel[blocks, threads](*args)
    cuda.synchronize()
    return (time.perf_counter() - start) / reps


def bandwidth_gbs(nbytes, seconds):
    return 2 * nbytes / seconds / 1e9

In [ ]:
# 6.2 CPU VS NAIVE GPU BENCHMARK

start = time.perf_counter()
B_cpu = transpose_cpu(A)
cpu_time = time.perf_counter() - start

gpu_time = time_gpu(transpose_gpu_naive, blocks_per_grid, threads_per_block, d_A, d_B)

speedup = cpu_time / gpu_time

print(f"CPU execution time:        {cpu_time:.6f} s  ({bandwidth_gbs(A.nbytes, cpu_time):.2f} GB/s)")
print(f"GPU naive kernel time:     {gpu_time:.6f} s  ({bandwidth_gbs(A.nbytes, gpu_time):.2f} GB/s, average of {REPS})")
print(f"GPU kernel-only speedup:   {speedup:.2f} x  (excludes RAM <-> VRAM transfer)")

transpose_results = pd.DataFrame({
    "Matrix Size": [N],
    "CPU Time (s)": [cpu_time],
    "GPU Naive Time (s)": [gpu_time],
    "GPU Kernel-only Speedup": [speedup],
    "CPU Correct": [cpu_correct],
    "GPU Correct": [gpu_correct],
})

transpose_results

In [ ]:
# 6.3 PERFORMANCE GRAPH

plt.figure(figsize=(8, 5))
bars = plt.bar(
    ['CPU (1 core)', 'GPU naive kernel'],
    [cpu_time, gpu_time],
    color=['steelblue', 'darkorange'],
)
plt.bar_label(bars, fmt='%.6f s')
plt.title(f'{N} x {N} Matrix Transposition: CPU vs Naive GPU')
plt.ylabel('Execution Time (seconds)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Optimized GPU Implementation: Shared-Memory Tiled Transpose

### The problem with the naive kernel

In the naive kernel, one side of every warp's memory access is scattered (Section 5). A warp of 32 threads that should need one or two memory transactions instead needs up to 32, and most of each fetched chunk is thrown away.

### The fix: transpose through shared memory

Each thread block handles one **32 × 32 tile** of the matrix:

1. **Load:** threads read the tile from `A` **row by row** (coalesced) into fast on-chip shared memory.
2. **Synchronize:** `cuda.syncthreads()` waits until the whole tile is loaded.
3. **Store:** threads read the tile from shared memory **column by column** and write it to `B` **row by row** (coalesced).

The scattered access now happens inside shared memory, which is on the chip and has no chunk penalty. Every access to VRAM is coalesced.

Following NVIDIA's design, each block has **32 × 8 threads**, so each thread moves 4 elements of its tile.

### Shared-memory bank conflicts

Shared memory is split into 32 **banks**, and each bank can serve one value per cycle. In a 32 × 32 tile, every element of a column falls in the **same bank**, so reading a column causes a 32-way **bank conflict** and the reads are done one after another. Making the tile **33** columns wide (one unused padding column) shifts each row by one bank, so a column is spread across all 32 banks.

Two tiled kernels are built to show this effect:

- **Tiled (bank conflicts):** tile is 32 × 32
- **Tiled (padded):** tile is 32 × 33, the final optimized kernel

In [ ]:
# 7.1 TILED KERNELS

TILE = 32
TILE_PADDED = TILE + 1   # one extra column removes shared-memory bank conflicts
TILED_THREADS = (32, 8)  # 32 x 8 threads cover a 32 x 32 tile, 4 elements per thread


@cuda.jit
def transpose_gpu_tiled_conflicts(A, B):
    tile = cuda.shared.array((TILE, TILE), dtype=float32)  # 32 x 32

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    rows, cols = A.shape

    # Load: read a tile of A row by row (coalesced)
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    for j in range(0, TILE, cuda.blockDim.y):
        if y + j < rows and x < cols:
            tile[ty + j, tx] = A[y + j, x]

    cuda.syncthreads()

    # Store: write the transposed tile to B row by row (coalesced)
    x = cuda.blockIdx.y * TILE + tx
    y = cuda.blockIdx.x * TILE + ty
    for j in range(0, TILE, cuda.blockDim.y):
        if y + j < cols and x < rows:
            B[y + j, x] = tile[tx, ty + j]


@cuda.jit
def transpose_gpu_tiled(A, B):
    tile = cuda.shared.array((TILE, TILE_PADDED), dtype=float32)  # 32 x 33

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    rows, cols = A.shape

    # Load: read a tile of A row by row (coalesced)
    x = cuda.blockIdx.x * TILE + tx
    y = cuda.blockIdx.y * TILE + ty
    for j in range(0, TILE, cuda.blockDim.y):
        if y + j < rows and x < cols:
            tile[ty + j, tx] = A[y + j, x]

    cuda.syncthreads()

    # Store: write the transposed tile to B row by row (coalesced)
    x = cuda.blockIdx.y * TILE + tx
    y = cuda.blockIdx.x * TILE + ty
    for j in range(0, TILE, cuda.blockDim.y):
        if y + j < cols and x < rows:
            B[y + j, x] = tile[tx, ty + j]


def tiled_grid(n):
    return ((n + TILE - 1) // TILE, (n + TILE - 1) // TILE)


print("Tile:", TILE, "x", TILE)
print("Threads per block:", TILED_THREADS)
print("Blocks per grid:", tiled_grid(N))

In [ ]:
# 7.2 VERIFY AND BENCHMARK: NAIVE VS OPTIMIZED

kernels = {
    'Naive': (transpose_gpu_naive, blocks_per_grid, threads_per_block),
    'Tiled (bank conflicts)': (transpose_gpu_tiled_conflicts, tiled_grid(N), TILED_THREADS),
    'Tiled (padded)': (transpose_gpu_tiled, tiled_grid(N), TILED_THREADS),
}

rows = []
for name, (kernel, blocks, threads) in kernels.items():
    t = time_gpu(kernel, blocks, threads, d_A, d_B)
    rows.append({
        'Kernel': name,
        'Correct': np.array_equal(d_B.copy_to_host(), B_reference),
        'Time (ms)': t * 1e3,
        'Effective BW (GB/s)': bandwidth_gbs(A.nbytes, t),
        'Speedup vs naive': gpu_time / t,
        'Speedup vs CPU': cpu_time / t,
    })

kernel_results = pd.DataFrame(rows)
tiled_conflict_time = kernel_results.loc[1, 'Time (ms)'] / 1e3
tiled_time = kernel_results.loc[2, 'Time (ms)'] / 1e3

if not kernel_results['Correct'].all():
    raise AssertionError('A tiled kernel produced a wrong result.')

kernel_results.round({'Time (ms)': 4, 'Effective BW (GB/s)': 1, 'Speedup vs naive': 2, 'Speedup vs CPU': 1})

In [ ]:
# 7.3 NAIVE VS OPTIMIZED GRAPH

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['darkorange', 'goldenrod', 'seagreen']
names = kernel_results['Kernel'].str.replace(' (', '\n(', regex=False)

bars = ax1.bar(names, kernel_results['Time (ms)'], color=colors)
ax1.bar_label(bars, fmt='%.3f ms')
ax1.set_ylabel('Kernel time (ms)')
ax1.set_title(f'{N} x {N}: kernel time (lower is better)')

bars = ax2.bar(names, kernel_results['Effective BW (GB/s)'], color=colors)
ax2.bar_label(bars, fmt='%.1f')
ax2.set_ylabel('Effective bandwidth (GB/s)')
ax2.set_title('Effective bandwidth (higher is better)')

for ax in (ax1, ax2):
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. CUDA Block and Grid Configuration

A kernel launch is described by two numbers:

- **Threads per block (block shape):** how many threads work together and can share memory. The GPU runs threads in groups of 32 called **warps**, and a block can have at most 1024 threads on the T4.
- **Blocks per grid:** how many blocks are needed to cover the matrix. It follows from the block shape and the matrix size.

The block shape decides which threads run in the same warp, and therefore which memory addresses a warp touches together. This experiment runs both kernels on the 4096 × 4096 matrix with different block shapes:

- **Naive kernel:** several 2D shapes from 64 to 1024 threads per block.
- **Tiled (padded) kernel:** the tile stays 32 × 32 and the number of thread rows changes, so each thread moves 32, 8, 4, 2 or 1 elements.

In [ ]:
# 8.1 BLOCK SHAPE EXPERIMENT

naive_configs = [(8, 8), (16, 16), (32, 8), (8, 32), (32, 16), (32, 32)]
tiled_configs = [(32, 1), (32, 4), (32, 8), (32, 16), (32, 32)]

rows = []
for kernel_name, kernel, configs in [
    ('Naive', transpose_gpu_naive, naive_configs),
    ('Tiled (padded)', transpose_gpu_tiled, tiled_configs),
]:
    for threads in configs:
        blocks = naive_grid(N, threads) if kernel_name == 'Naive' else tiled_grid(N)
        t = time_gpu(kernel, blocks, threads, d_A, d_B)
        rows.append({
            'Kernel': kernel_name,
            'Threads per block': f'{threads[0]} x {threads[1]}',
            'Threads in block': threads[0] * threads[1],
            'Blocks per grid': f'{blocks[0]} x {blocks[1]}',
            'Correct': np.array_equal(d_B.copy_to_host(), B_reference),
            'Time (ms)': t * 1e3,
            'Effective BW (GB/s)': bandwidth_gbs(A.nbytes, t),
        })

block_results = pd.DataFrame(rows)
block_results.round({'Time (ms)': 4, 'Effective BW (GB/s)': 1})

In [ ]:
# 8.2 BLOCK SHAPE GRAPH

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, (kernel_name, color) in zip(axes, [('Naive', 'darkorange'), ('Tiled (padded)', 'seagreen')]):
    data = block_results[block_results['Kernel'] == kernel_name]
    bars = ax.bar(data['Threads per block'], data['Effective BW (GB/s)'], color=color)
    ax.bar_label(bars, fmt='%.0f')
    ax.set_title(f'{kernel_name} kernel')
    ax.set_xlabel('Threads per block (x by y)')
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Effective bandwidth (GB/s)')
fig.suptitle(f'Effect of block shape, {N} x {N} matrix')
plt.tight_layout()
plt.show()

## 9. Matrix Size Benchmark

This experiment repeats the CPU, naive GPU and tiled GPU transposes on matrices from 256 × 256 to 8192 × 8192, to see how execution time and speedup change with size.

- CPU time is the **best of 3 runs**, to reduce noise at small sizes.
- GPU times are kernel-only averages, as in Section 6.
- Each new matrix size has 4 times as many elements as the previous one, so an ideal transpose would take about 4 times longer at each step.

In [ ]:
# 9.1 MATRIX SIZE SWEEP

SIZES = [256, 512, 1024, 2048, 4096, 8192]
rng = np.random.default_rng(42)

rows = []
for n in SIZES:
    M = rng.random((n, n), dtype=np.float32)

    cpu_runs = []
    for _ in range(3):
        start = time.perf_counter()
        transpose_cpu(M)
        cpu_runs.append(time.perf_counter() - start)
    t_cpu = min(cpu_runs)

    d_M = cuda.to_device(M)
    d_Mt = cuda.device_array_like(d_M)
    t_naive = time_gpu(transpose_gpu_naive, naive_grid(n, threads_per_block), threads_per_block, d_M, d_Mt)
    t_tiled = time_gpu(transpose_gpu_tiled, tiled_grid(n), TILED_THREADS, d_M, d_Mt)

    rows.append({
        'Matrix size': n,
        'Elements': n * n,
        'CPU (s)': t_cpu,
        'GPU naive (s)': t_naive,
        'GPU tiled (s)': t_tiled,
        'Speedup (naive)': t_cpu / t_naive,
        'Speedup (tiled)': t_cpu / t_tiled,
        'CPU BW (GB/s)': bandwidth_gbs(M.nbytes, t_cpu),
        'GPU tiled BW (GB/s)': bandwidth_gbs(M.nbytes, t_tiled),
    })

del M, d_M, d_Mt  # free memory before the large test

sweep_results = pd.DataFrame(rows)
sweep_results.round({
    'CPU (s)': 6, 'GPU naive (s)': 6, 'GPU tiled (s)': 6,
    'Speedup (naive)': 1, 'Speedup (tiled)': 1,
    'CPU BW (GB/s)': 2, 'GPU tiled BW (GB/s)': 1,
})

In [ ]:
# 9.2 EXECUTION TIME AND SPEEDUP VS MATRIX SIZE

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for column, color, label in [
    ('CPU (s)', 'steelblue', 'CPU (1 core)'),
    ('GPU naive (s)', 'darkorange', 'GPU naive'),
    ('GPU tiled (s)', 'seagreen', 'GPU tiled (padded)'),
]:
    ax1.plot(sweep_results['Matrix size'], sweep_results[column], 'o-', color=color, label=label)

ax1.set_xscale('log', base=2)
ax1.set_yscale('log')
ax1.set_xlabel('Matrix size N (N x N)')
ax1.set_ylabel('Time (seconds, log scale)')
ax1.set_title('Execution time vs matrix size')
ax1.legend()

for column, color, label in [
    ('Speedup (naive)', 'darkorange', 'GPU naive'),
    ('Speedup (tiled)', 'seagreen', 'GPU tiled (padded)'),
]:
    ax2.plot(sweep_results['Matrix size'], sweep_results[column], 'o-', color=color, label=label)

ax2.set_xscale('log', base=2)
ax2.set_xlabel('Matrix size N (N x N)')
ax2.set_ylabel('Speedup over CPU (x)')
ax2.set_title('GPU kernel-only speedup vs matrix size')
ax2.legend()

for ax in (ax1, ax2):
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZES)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Comparison with Published Results

To check how this work differs from existing results, the measurements are compared with:

1. **NVIDIA's published transpose benchmark** – Mark Harris, *An Efficient Matrix Transpose in CUDA C/C++*, NVIDIA Developer Blog. Tesla M2050 (Fermi) and Tesla K20c (Kepler), 1024 × 1024 `float32`.
2. **A published CPU transpose benchmark** – *Why a Mathematically Free Matrix Transpose Is 10× Slower on CPUs* (besthub.dev). Naive vs cache-blocked C code, 1024 × 1024 `float64`.

NVIDIA's blog follows the same path as this project, and its kernels map directly onto ours:

| NVIDIA kernel | This project |
|---|---|
| copy | copy kernel (below) |
| transposeNaive | Naive (Section 5) |
| transposeCoalesced | Tiled (bank conflicts) (Section 7) |
| transposeNoBankConflicts | Tiled (padded) (Section 7) |

NVIDIA also runs a plain **copy kernel**. It moves the same amount of data with perfect access patterns, so it is the practical speed limit for a transpose. Expressing each transpose as a **percentage of copy** lets results from different GPUs be compared fairly, even though their peak bandwidths differ (Tesla T4 peak: 320 GB/s). Raw GB/s values are also shown, but newer hardware is naturally faster.

**Sources**
- https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/
- https://www.besthub.dev/articles/why-a-mathematically-free-matrix-transpose-is-10-slower-on-cpus-73363c8650d8
- NVIDIA T4 product brief (PB-09256-001): peak memory bandwidth up to 320 GB/s

In [ ]:
# 10.1 COPY KERNEL AND COMPARISON TABLE

T4_PEAK_GBS = 320  # NVIDIA T4 product brief


# Copy kernel: same data movement as a transpose, fully coalesced.
# x (fastest-changing thread index) maps to the last array index.
@cuda.jit
def copy_gpu(A, B):
    x, y = cuda.grid(2)
    if y < A.shape[0] and x < A.shape[1]:
        B[y, x] = A[y, x]


d_C = cuda.device_array_like(d_A)
copy_time = time_gpu(copy_gpu, naive_grid(N, (32, 8)), (32, 8), d_A, d_C)
print('Copy kernel correct:', np.array_equal(d_C.copy_to_host(), A))
del d_C

copy_bw = bandwidth_gbs(A.nbytes, copy_time)
naive_bw = bandwidth_gbs(A.nbytes, gpu_time)
conflict_bw = bandwidth_gbs(A.nbytes, tiled_conflict_time)
tiled_bw = bandwidth_gbs(A.nbytes, tiled_time)
cpu_bw = bandwidth_gbs(A.nbytes, cpu_time)

# GPU results in GB/s (NVIDIA blog values: 1024 x 1024 float32, ECC enabled)
gpu_bw = {
    'Tesla M2050': {'copy': 105.2, 'naive': 18.8, 'coalesced': 51.3, 'no bank conflicts': 99.5},
    'Tesla K20c': {'copy': 136.0, 'naive': 55.3, 'coalesced': 97.6, 'no bank conflicts': 144.3},
    'Tesla T4': {'copy': copy_bw, 'naive': naive_bw, 'coalesced': conflict_bw, 'no bank conflicts': tiled_bw},
}
our_names = {'copy': 'copy', 'naive': 'naive', 'coalesced': 'tiled (bank conflicts)', 'no bank conflicts': 'tiled (padded)'}

# Published CPU results (besthub.dev article, 1024 x 1024 float64, seconds)
cpu_ref_bytes = 1024 * 1024 * 8
cpu_ref_times = {
    'Naive': 31.4e-3,
    'Blocked 8x8': 5.2e-3,
    'Blocked 8x8 + SIMD': 3.1e-3,
    'Blocked 32x32': 2.8e-3,
}

rows = []
for gpu_label, r in gpu_bw.items():
    ours = gpu_label == 'Tesla T4'
    for kernel_name, bw in r.items():
        rows.append([
            'This project' if ours else 'NVIDIA blog',
            gpu_label,
            'GPU ' + (our_names[kernel_name] if ours else kernel_name),
            f'{N} x {N}' if ours else '1024 x 1024',
            bw,
            100 * bw / r['copy'],
        ])

for name, t in cpu_ref_times.items():
    rows.append(['CPU article', 'CPU (not stated)', f'CPU {name}', '1024 x 1024', bandwidth_gbs(cpu_ref_bytes, t), None])
rows.append(['This project', 'Colab CPU', 'CPU naive (1 core)', f'{N} x {N}', cpu_bw, None])

comparison = pd.DataFrame(rows, columns=['Source', 'Hardware', 'Implementation', 'Matrix', 'Effective BW (GB/s)', '% of copy'])

print(f'T4 copy kernel: {copy_bw:.1f} GB/s ({100 * copy_bw / T4_PEAK_GBS:.0f}% of the {T4_PEAK_GBS} GB/s peak)')
comparison.round({'Effective BW (GB/s)': 2, '% of copy': 1})

In [ ]:
# 10.2 GPU COMPARISON GRAPHS: NVIDIA VS THIS PROJECT

kernel_keys = ['naive', 'coalesced', 'no bank conflicts']
kernel_labels = ['Naive', 'Tiled / coalesced\n(bank conflicts)', 'Tiled, padded\n(no bank conflicts)']
gpu_colors = {'Tesla M2050': 'silver', 'Tesla K20c': 'gray', 'Tesla T4': 'darkorange'}
x = np.arange(len(kernel_keys))
width = 0.27

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i, (gpu_label, r) in enumerate(gpu_bw.items()):
    legend = gpu_label + (' (this project)' if gpu_label == 'Tesla T4' else ' (NVIDIA)')
    pct = [100 * r[k] / r['copy'] for k in kernel_keys]
    bars = ax1.bar(x + (i - 1) * width, pct, width, color=gpu_colors[gpu_label], label=legend)
    ax1.bar_label(bars, fmt='%.0f%%', fontsize=8)
    bars = ax2.bar(x + (i - 1) * width, [r[k] for k in kernel_keys], width, color=gpu_colors[gpu_label], label=legend)
    ax2.bar_label(bars, fmt='%.0f', fontsize=8)

ax1.axhline(100, color='black', linestyle=':', linewidth=1)
ax1.set_ylabel('% of copy kernel bandwidth')
ax1.set_title('Transpose as % of copy speed (fair across GPUs)')

ax2.axhline(T4_PEAK_GBS, color='darkorange', linestyle='--', linewidth=1)
ax2.text(len(kernel_keys) - 0.5, T4_PEAK_GBS, f'T4 peak ({T4_PEAK_GBS} GB/s)', ha='right', va='bottom', color='darkorange')
ax2.set_ylim(0, T4_PEAK_GBS * 1.15)
ax2.set_ylabel('Effective bandwidth (GB/s)')
ax2.set_title('Effective bandwidth (raw)')

for ax in (ax1, ax2):
    ax.set_xticks(x)
    ax.set_xticklabels(kernel_labels)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 10.3 CPU COMPARISON GRAPH

cpu_labels = [name.replace(' + ', '\n+ ') for name in cpu_ref_times] + ['This project\n(naive, 1 core)']
cpu_vals = [bandwidth_gbs(cpu_ref_bytes, t) for t in cpu_ref_times.values()] + [cpu_bw]

plt.figure(figsize=(9, 4.5))
bars = plt.bar(cpu_labels, cpu_vals, color=['gray'] * len(cpu_ref_times) + ['steelblue'])
plt.bar_label(bars, fmt='%.2f')
plt.title('CPU transpose effective bandwidth: published article (gray) vs this project (blue)')
plt.ylabel('Effective bandwidth (GB/s)')
plt.xticks(fontsize=8)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 10.4 How This Work Differs from the Published Results

1. **Same optimization path, same pattern.** Both NVIDIA and this project go naive → tiled → tiled with padding. Compare the "% of copy" bars: if the T4 shows the same order (each step faster than the last), the optimizations behave on Turing as they did on Fermi and Kepler.

2. **Reversed access pattern in the naive kernel.** Here, neighbouring threads read *down a column* of `A` (scattered reads) and write *along a row* of `B` (coalesced writes). NVIDIA's naive kernel does the opposite: coalesced reads and scattered writes. Scattered reads can be partly served by the L1/L2 caches, while scattered writes cannot be combined, so this naive kernel is expected to do relatively better than NVIDIA's.

3. **Newer GPU architecture.** The Tesla T4 (Turing, 2018, 320 GB/s, 4 MB L2 cache) has much larger caches than the Fermi M2050 and Kepler K20c. NVIDIA's own numbers already show the naive penalty shrinking from Fermi (18% of copy) to Kepler (41% of copy).

4. **Larger matrix.** This project uses 4096 × 4096 (64 MB); NVIDIA used 1024 × 1024 (4 MB). The larger matrix is far bigger than any cache, so the result reflects real VRAM bandwidth.

5. **Thread-block design.** The tiled kernels use NVIDIA's design (32 × 32 tiles, 32 × 8 threads). The naive kernel uses 16 × 16 blocks with one element per thread. Section 8 shows how much this choice matters.

6. **Timing method.** Like NVIDIA, GPU times are averages of repeated launches, not single launches.

7. **CPU baseline is a naive, single-core loop.** The published CPU results show cache blocking making a transpose about 6–11× faster than the naive loop. The CPU result here is in the naive range, which partly reflects different hardware, and possibly also the 4096 size: each column step jumps exactly 16 KB, a power of two, which can cause cache conflicts. GPU speedups in this notebook are therefore measured against an *unoptimized* CPU loop. Against a blocked, multi-core CPU version, the gap would be much smaller.

## 11. Full-Capacity Test: Large Matrix, Memory Usage and Data Transfer

### 11.1 Capacity planning

Before the large test, this step estimates the largest square matrix the current Colab session can safely hold, from the RAM and VRAM that are free. It does **not** allocate the large matrix.

A `float32` element uses 4 bytes. The estimate reserves room for three host matrices (input, one result, and temporary memory for the correctness check) and two GPU matrices (input and output). It uses only 60% of available RAM and 70% of free VRAM, to leave room for Colab, CUDA and Numba.

Run time is projected by assuming time grows with $N^2$ (the number of elements), scaling from the 4096 × 4096 results. This is a planning estimate, and it is checked against the measured times later. Colab resources can change between sessions; see the [Google Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
# 11.1 CAPACITY PLANNING AND RUNTIME ESTIMATE

BYTES_PER_FLOAT32 = np.dtype(np.float32).itemsize
HOST_MATRIX_COPIES = 3
GPU_MATRIX_COPIES = 2
RAM_SAFETY_FRACTION = 0.60
VRAM_SAFETY_FRACTION = 0.70
ALIGNMENT = 256

available_ram_bytes = psutil.virtual_memory().available
free_vram_bytes, total_vram_bytes = cuda.current_context().get_memory_info()

max_n_from_ram = int(np.sqrt(
    available_ram_bytes * RAM_SAFETY_FRACTION / (HOST_MATRIX_COPIES * BYTES_PER_FLOAT32)
))
max_n_from_vram = int(np.sqrt(
    free_vram_bytes * VRAM_SAFETY_FRACTION / (GPU_MATRIX_COPIES * BYTES_PER_FLOAT32)
))


def round_down(value, multiple):
    return (value // multiple) * multiple


recommended_max_n = round_down(min(max_n_from_ram, max_n_from_vram), ALIGNMENT)
suggested_large_test_n = round_down(int(recommended_max_n * 0.80), ALIGNMENT)

if suggested_large_test_n < N:
    raise RuntimeError(
        'The current session does not have enough free memory for a larger test. Restart the runtime and rerun the notebook.'
    )


def matrix_gib(side_length):
    return side_length ** 2 * BYTES_PER_FLOAT32 / (1024 ** 3)


def estimated_seconds(side_length, baseline_seconds):
    return baseline_seconds * (side_length / N) ** 2


capacity_plan = pd.DataFrame([
    {
        'Plan': label,
        'Matrix size': f'{n} x {n}',
        'Elements': n ** 2,
        'One matrix (GiB)': matrix_gib(n),
        'Estimated CPU time (s)': estimated_seconds(n, cpu_time),
        'Estimated GPU naive kernel (s)': estimated_seconds(n, gpu_time),
        'Estimated GPU tiled kernel (s)': estimated_seconds(n, tiled_time),
    }
    for label, n in [
        ('Measured baseline', N),
        ('Suggested large test (80% of limit)', suggested_large_test_n),
        ('Conservative capacity limit', recommended_max_n),
    ]
])

print(f'Available system RAM: {available_ram_bytes / 1024 ** 3:.2f} GiB')
print(f'Free GPU VRAM: {free_vram_bytes / 1024 ** 3:.2f} GiB of {total_vram_bytes / 1024 ** 3:.2f} GiB')
print(f'RAM-limited maximum side length: {max_n_from_ram:,}')
print(f'VRAM-limited maximum side length: {max_n_from_vram:,}')
print('')
capacity_plan.round({
    'One matrix (GiB)': 2,
    'Estimated CPU time (s)': 2,
    'Estimated GPU naive kernel (s)': 4,
    'Estimated GPU tiled kernel (s)': 4,
})

### 11.2 Final test

This is the final benchmark, run at the size suggested above (80% of the conservative memory limit). It measures:

- **CPU, single core:** the Section 4 code.
- **CPU, all cores:** a parallel version using `@njit(parallel=True)` and `prange`. The Section 4 code uses only one core, so this is what actually uses the full CPU. Colab usually provides only 2 virtual CPUs, so expect a modest gain.
- **GPU naive and tiled kernels:** kernel-only, averaged over `GPU_REPS` launches.
- **RAM → VRAM and VRAM → RAM transfers:** the time to move the matrix over the PCIe bus. Adding these to the kernel time gives the **end-to-end** GPU time.
- **Memory usage:** RAM used by the notebook process and VRAM used by the test matrices.

What "full capacity" means here: a bigger matrix fills **memory**, not compute. The GPU is already fully busy at 4096 × 4096 (tens of thousands of thread blocks spread over 40 SMs). The large test shows how performance holds up when far more data moves than any cache can hold.

To push closer to the limit, set `FINAL_N = recommended_max_n` in 11.3. This needs more RAM and may crash the session.

In [ ]:
# 11.3 FINAL TEST SETUP

FINAL_N = suggested_large_test_n   # set to recommended_max_n to push closer to the limit
GPU_REPS = 10


@njit(parallel=True)
def transpose_cpu_parallel(A):
    rows, cols = A.shape
    B = np.empty((cols, rows), dtype=A.dtype)
    for i in prange(rows):
        for j in range(cols):
            B[j, i] = A[i, j]
    return B


transpose_cpu_parallel(A)  # compile once on the 4096 x 4096 matrix

# Generate float32 directly, so no temporary float64 copy uses extra RAM
A_big = rng.random((FINAL_N, FINAL_N), dtype=np.float32)

print(f'Matrix: {FINAL_N} x {FINAL_N} float32 ({A_big.nbytes / 1024 ** 3:.2f} GiB per matrix)')
print('CPU threads used by Numba:', get_num_threads())
print('Naive grid:', naive_grid(FINAL_N, threads_per_block), 'blocks of', threads_per_block)
print('Tiled grid:', tiled_grid(FINAL_N), 'blocks of', TILED_THREADS)

In [ ]:
# 11.4 RUN THE FINAL TEST

process = psutil.Process()
final = {}

# CPU, single core
start = time.perf_counter()
B = transpose_cpu(A_big)
final['CPU single-core'] = time.perf_counter() - start
cpu_single_ok = np.array_equal(B, A_big.T)
del B

# CPU, all cores
start = time.perf_counter()
B = transpose_cpu_parallel(A_big)
final['CPU all cores'] = time.perf_counter() - start
cpu_parallel_ok = np.array_equal(B, A_big.T)
ram_in_use = process.memory_info().rss  # input and result both in RAM
del B

# GPU: copy input RAM -> VRAM
free_before, _ = cuda.current_context().get_memory_info()
start = time.perf_counter()
d_A_big = cuda.to_device(A_big)
cuda.synchronize()
h2d_time = time.perf_counter() - start

d_B_big = cuda.device_array_like(d_A_big)
free_after, _ = cuda.current_context().get_memory_info()
vram_used = free_before - free_after

# GPU: kernels only
final['GPU naive kernel'] = time_gpu(
    transpose_gpu_naive, naive_grid(FINAL_N, threads_per_block), threads_per_block, d_A_big, d_B_big, reps=GPU_REPS
)
gpu_naive_ok = np.array_equal(d_B_big.copy_to_host(), A_big.T)

final['GPU tiled kernel'] = time_gpu(
    transpose_gpu_tiled, tiled_grid(FINAL_N), TILED_THREADS, d_A_big, d_B_big, reps=GPU_REPS
)

# GPU: copy result VRAM -> RAM
start = time.perf_counter()
B = d_B_big.copy_to_host()
d2h_time = time.perf_counter() - start
gpu_tiled_ok = np.array_equal(B, A_big.T)
del B, d_A_big, d_B_big

final['GPU tiled incl. transfers'] = h2d_time + final['GPU tiled kernel'] + d2h_time

print('CPU single-core correct:', cpu_single_ok)
print('CPU all cores correct:  ', cpu_parallel_ok)
print('GPU naive correct:      ', gpu_naive_ok)
print('GPU tiled correct:      ', gpu_tiled_ok)

In [ ]:
# 11.5 MEMORY USAGE AND DATA TRANSFER

memory_usage = pd.DataFrame([
    ['One matrix (GiB)', A_big.nbytes / 1024 ** 3],
    ['Notebook RAM in use during CPU test (GiB)', ram_in_use / 1024 ** 3],
    ['VRAM used by input + output (GiB)', vram_used / 1024 ** 3],
    ['RAM -> VRAM transfer time (s)', h2d_time],
    ['RAM -> VRAM transfer speed (GB/s)', A_big.nbytes / h2d_time / 1e9],
    ['VRAM -> RAM transfer time (s)', d2h_time],
    ['VRAM -> RAM transfer speed (GB/s)', A_big.nbytes / d2h_time / 1e9],
    ['GPU tiled kernel time (s)', final['GPU tiled kernel']],
    ['Transfers as % of end-to-end GPU time', 100 * (h2d_time + d2h_time) / final['GPU tiled incl. transfers']],
], columns=['Measurement', 'Value'])

memory_usage.round({'Value': 3})

In [ ]:
# 11.6 FINAL RESULTS

estimates = {
    'CPU single-core': estimated_seconds(FINAL_N, cpu_time),
    'GPU naive kernel': estimated_seconds(FINAL_N, gpu_time),
    'GPU tiled kernel': estimated_seconds(FINAL_N, tiled_time),
}

final_results = pd.DataFrame([
    {
        'Implementation': name,
        'Time (s)': t,
        'Effective BW (GB/s)': bandwidth_gbs(A_big.nbytes, t),
        'Speedup vs CPU single-core': final['CPU single-core'] / t,
        'Section 11.1 estimate (s)': estimates.get(name),
    }
    for name, t in final.items()
])

print(f'Final test: {FINAL_N} x {FINAL_N} float32')
final_results.round({
    'Time (s)': 4,
    'Effective BW (GB/s)': 2,
    'Speedup vs CPU single-core': 1,
    'Section 11.1 estimate (s)': 4,
})

In [ ]:
# 11.7 FINAL TEST GRAPH

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'royalblue', 'darkorange', 'seagreen', 'darkgreen']
names = final_results['Implementation'].str.replace(' ', '\n', n=1)

bars = ax1.bar(names, final_results['Time (s)'], color=colors)
ax1.bar_label(bars, fmt='%.3f s', fontsize=8)
ax1.set_yscale('log')
ax1.set_ylabel('Time (seconds, log scale)')
ax1.set_title(f'{FINAL_N} x {FINAL_N} transpose: execution time')

bars = ax2.bar(names, final_results['Speedup vs CPU single-core'], color=colors)
ax2.bar_label(bars, fmt='%.1fx', fontsize=8)
ax2.set_yscale('log')
ax2.set_ylabel('Speedup (x, log scale)')
ax2.set_title('Speedup vs single-core CPU')

for ax in (ax1, ax2):
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 11.8 Reading the final results

- **Kernel only vs including transfers.** The PCIe 3.0 x16 bus that connects the T4 has a theoretical limit of about 16 GB/s, far below the 320 GB/s of VRAM. If the transfers take much longer than the kernel, the GPU only pays off when the data already lives in VRAM. This is the normal case in AI workloads, where tensors stay on the GPU between operations.
- **CPU all cores vs single core.** The gain is limited by Colab's small number of CPU cores and by RAM bandwidth, since the CPU transpose is also memory-bound.
- **Measured time vs Section 11.1 estimate.** If they are close, the $N^2$ scaling assumption holds. If the CPU is much slower than estimated, cache misses grow worse at large sizes.
- **Speedup at 4096 vs at the final size.** Compare with Sections 6 and 9 to see whether the GPU advantage grows or shrinks with matrix size.

## 12. Conclusions, Future Work and CSV Export

### 12.1 Key numbers from this run

The cell below pulls the main results from every section into one summary, so the conclusions match the numbers measured in this session.

In [ ]:
# 12.1 KEY NUMBERS FROM THIS RUN

naive_cfg = block_results[block_results['Kernel'] == 'Naive']
tiled_cfg = block_results[block_results['Kernel'] == 'Tiled (padded)']
first, last = sweep_results.iloc[0], sweep_results.iloc[-1]
fastest_naive = naive_cfg.loc[naive_cfg['Time (ms)'].idxmin()]
slowest_naive = naive_cfg.loc[naive_cfg['Time (ms)'].idxmax()]
fastest_tiled = tiled_cfg.loc[tiled_cfg['Time (ms)'].idxmin()]
slowest_tiled = tiled_cfg.loc[tiled_cfg['Time (ms)'].idxmax()]
end_to_end_speedup = final['CPU single-core'] / final['GPU tiled incl. transfers']
kernel_speedup = final['CPU single-core'] / final['GPU tiled kernel']

summary = [
    f"Optimization: at {N} x {N}, the tiled (padded) kernel took {tiled_time * 1e3:.3f} ms vs "
    f"{gpu_time * 1e3:.3f} ms for the naive kernel ({gpu_time / tiled_time:.2f}x faster).",

    f"Bandwidth: the tiled kernel reached {tiled_bw:.0f} GB/s, which is {100 * tiled_bw / copy_bw:.0f}% "
    f"of the T4 copy kernel and {100 * tiled_bw / T4_PEAK_GBS:.0f}% of the {T4_PEAK_GBS} GB/s peak.",

    f"Bank conflicts: padding the shared tile changed the kernel time from {tiled_conflict_time * 1e3:.3f} ms "
    f"to {tiled_time * 1e3:.3f} ms.",

    f"Block shape: the naive kernel ranged from {fastest_naive['Time (ms)']:.3f} ms ({fastest_naive['Threads per block']}) "
    f"to {slowest_naive['Time (ms)']:.3f} ms ({slowest_naive['Threads per block']}); the tiled kernel from "
    f"{fastest_tiled['Time (ms)']:.3f} ms ({fastest_tiled['Threads per block']}) to "
    f"{slowest_tiled['Time (ms)']:.3f} ms ({slowest_tiled['Threads per block']}).",

    f"Matrix size: the tiled GPU speedup over the CPU went from {first['Speedup (tiled)']:.1f}x at "
    f"{SIZES[0]} x {SIZES[0]} to {last['Speedup (tiled)']:.1f}x at {SIZES[-1]} x {SIZES[-1]}.",

    f"Published results: NVIDIA's padded kernel reached about 95-106% of copy; this project's reached "
    f"{100 * tiled_bw / copy_bw:.0f}%. The naive kernel reached {100 * naive_bw / copy_bw:.0f}% of copy "
    f"(NVIDIA: 18% on M2050, 41% on K20c).",

    f"Data transfer: at {FINAL_N} x {FINAL_N}, the tiled kernel took {final['GPU tiled kernel']:.4f} s, but "
    f"moving data in and out took {h2d_time + d2h_time:.3f} s. The speedup over one CPU core falls from "
    f"{kernel_speedup:.0f}x (kernel only) to {end_to_end_speedup:.1f}x (including transfers).",

    f"CPU cores: using all {get_num_threads()} Numba threads was "
    f"{final['CPU single-core'] / final['CPU all cores']:.2f}x faster than one core.",
]

for i, line in enumerate(summary, 1):
    print(f'{i}. {line}\n')

In [ ]:
# 12.2 CSV EXPORT

os.makedirs('results', exist_ok=True)

tables = {
    '06_initial_benchmark': transpose_results,
    '07_naive_vs_tiled': kernel_results,
    '08_block_configurations': block_results,
    '09_matrix_size_sweep': sweep_results,
    '10_published_comparison': comparison,
    '11_capacity_plan': capacity_plan,
    '11_memory_and_transfer': memory_usage,
    '11_final_test': final_results,
}

for name, df in tables.items():
    df.to_csv(f'results/{name}.csv', index=False)

shutil.make_archive('transpose_results', 'zip', 'results')

print(f'Saved {len(tables)} CSV files to results/ and transpose_results.zip')
print("Download from Colab's Files panel (folder icon on the left).")

### 12.3 Conclusions

1. **Matrix transposition is memory-bound.** It performs no arithmetic, so its speed is set by how fast memory can move data. Effective bandwidth (GB/s) describes its performance better than raw time, and the copy kernel sets the practical upper limit.

2. **Memory access pattern matters more than raw parallelism.** The naive GPU kernel already uses thousands of threads, but half of its memory accesses are scattered. Staging each tile in shared memory makes every VRAM access coalesced, which is what the tiled kernel's higher bandwidth reflects (summary points 1–2).

3. **Shared memory has its own hardware details.** A 32 × 32 tile causes bank conflicts when columns are read; padding the tile to 32 × 33 removes them at the cost of one unused column (summary point 3).

4. **Block configuration affects performance** because it decides which threads share a warp and therefore which addresses are fetched together (summary point 4, Section 8).

5. **The GPU advantage depends on matrix size.** Small matrices give the GPU too little work to hide launch overhead, while large matrices keep all 40 SMs busy (summary point 5, Section 9).

6. **The results follow the same pattern as NVIDIA's published benchmark**, on newer hardware with larger caches (summary point 6, Section 10).

7. **Data transfer can outweigh the computation.** Moving the matrix over PCIe is much slower than transposing it inside VRAM, so the GPU is most useful when data stays on the GPU across many operations, as in AI and machine learning workloads (summary point 7, Section 11).

8. **A speedup figure depends on what it is compared against.** The CPU baseline here is a naive loop, and the CPU gains only modestly from extra cores on Colab (summary point 8). Speedups should always be read together with the baseline and with whether transfers are included.

### 12.4 Future work

- **Optimized CPU baseline:** a cache-blocked, multi-core CPU transpose, for a fairer CPU vs GPU comparison.
- **Faster transfers:** pinned (page-locked) host memory and CUDA streams to overlap transfers with computation.
- **Profiling:** NVIDIA Nsight Compute to measure actual DRAM throughput, cache hit rates and bank conflicts directly.
- **Other shapes and types:** rectangular matrices, `float64` and `float16`, and in-place transposition.
- **Other hardware:** repeat the benchmark on newer Colab GPUs (such as L4 or A100) and compare scaling across architectures.
- **Library comparison:** compare against CuPy and PyTorch transposes, which use NVIDIA's own optimized kernels.